# Does the Most Accurate Forecast Give the Lowest Inventory Cost? — Spare-Parts Demand (final)

**Research question:** For intermittent spare-parts demand, does the forecasting method with the best accuracy also give the lowest inventory cost? And does stocking by part criticality reduce cost?

**Main dataset (real):** RAF spare-parts demand — 5,000 items, 84 months (Jan 1996 – Dec 2002), with each item's **real lead time and unit price**. **Second dataset (real, industrial):** OIL — about 14,500 materials of an industrial site, 56 months, with delivery time and price. Optional third: Car Parts (Monash/Zenodo). All are public.
Data source for RAF and OIL: the `danieldehaan96/spdf` GitHub repository (cite it; it has a CITATION.cff). That repository comes from a master's thesis that also benchmarked these methods — **cite it in your related work and state your difference** (accuracy-vs-cost link + criticality-based stocking).

**Pipeline:** load data → classify demand (ADI / CV²) → forecast (MA, Croston, SBA, TSB, LightGBM, Adaptive) → rolling evaluation (MASE, RMSSE, bias) → month-by-month inventory simulation using each item's own lead time and price → accuracy vs cost → **cost at equal service level (fill rate)** → sensitivity.

**Honest limitations (put these in your report):**
- RAF data are 1996–2002 and look like general stores/tools; OIL has no demand-size units documented. Neither is guaranteed to represent modern machine parts.
- Criticality classes and shortage costs are **assumptions** (no such data exist); holding cost is an assumed % of the real unit price.
- Novelty is the fair comparison of accuracy vs cost, not a new algorithm.

## 0. Settings and imports

In [ ]:
!pip install lightgbm xlrd -q

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import os, zipfile, urllib.request, warnings
from scipy.stats import norm, wilcoxon
import lightgbm as lgb
warnings.filterwarnings('ignore')

# ---------------- SETTINGS (change here, nowhere else) ----------------
SEED        = 42
DATASET     = 'RAF'   # 'RAF' (main), 'OIL' (second real industrial set), or 'CARPARTS' (optional; no price/lead time)
TEST_MONTHS = 12      # last 12 months held out (time-based split)
CAL_MONTHS  = 12      # last 12 TRAIN months used to estimate forecast-error spread (sigma)
MIN_DEMANDS = 4       # keep series with at least this many non-zero months in TRAIN
SAMPLE_N    = 2000    # random sample of series for speed; set None to use all
LEAD_TIME   = 2       # months; used ONLY when the dataset has no lead times (CARPARTS / synthetic)
HOLD_RATE   = 0.20    # holding cost per YEAR as a fraction of unit price (assumption)
SHORT_MULT  = {'Critical': 1.0, 'Essential': 0.3, 'Desirable': 0.1}   # shortage cost per unit short, as a multiple of unit price (assumption)
CLASS_SHARE = {'Critical': 0.2, 'Essential': 0.3, 'Desirable': 0.5}    # share of parts per class (assumption)
Z_FLAT      = 1.65    # one-size-fits-all service factor (~95%)
# -----------------------------------------------------------------------
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
print('Ready')

## 1. Load data
- **RAF** (default): real, 5,000 items, with lead time (months) and price (£) per item.
- **CARPARTS**: real, 2,674 monthly series, no price / lead time (uses `LEAD_TIME` and price = 1).
If a download fails the notebook tries the other real dataset, then a clearly labelled **synthetic fallback** — never report fallback results as real.

In [ ]:
RAF_URL = 'https://raw.githubusercontent.com/danieldehaan96/spdf/main/RAF%20data%20-%207%20years%20demand%20%20-%205000%20items.xls'
CAR_URL = 'https://zenodo.org/records/4656021/files/car_parts_dataset_without_missing_values.zip?download=1'

def load_raf():
    urllib.request.urlretrieve(RAF_URL, 'raf.xls')
    d = pd.read_excel('raf.xls')
    lead  = pd.to_numeric(d.iloc[:, 2], errors='coerce').fillna(LEAD_TIME).values.astype(float)   # months
    price = pd.to_numeric(d.iloc[:, 3], errors='coerce').fillna(0).values.astype(float)           # GBP
    Y = d.iloc[:, 4:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(float)
    ok = price > 0                                    # cost needs a positive price
    return Y[ok], price[ok], lead[ok]

OIL_URL = 'https://raw.githubusercontent.com/danieldehaan96/spdf/main/OIL.xls'

def load_oil():
    urllib.request.urlretrieve(OIL_URL, 'oil.xls')
    dem  = pd.read_excel('oil.xls', sheet_name='GOEIE')        # col 0 = Material, cols 1..56 = months, last = price
    info = pd.read_excel('oil.xls', sheet_name='Data 1a')      # Material, Del. time (calendar days), Price ...
    info = info[pd.to_numeric(info.iloc[:, 0], errors='coerce').notna()].copy()
    info['mat'] = pd.to_numeric(info.iloc[:, 0]).astype(int)
    lead_days = pd.to_numeric(info.drop_duplicates('mat').set_index('mat')['Del. time'], errors='coerce')
    mat = pd.to_numeric(dem.iloc[:, 0], errors='coerce')
    ok = mat.notna()
    dem = dem[ok]; mat = mat[ok].astype(int).values
    Y = dem.iloc[:, 1:-1].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(float)
    price = pd.to_numeric(dem.iloc[:, -1], errors='coerce').fillna(0).values.astype(float)
    lead = (lead_days.reindex(mat).fillna(LEAD_TIME * 30).values.astype(float)) / 30.0     # days -> months
    good = price > 0
    return Y[good], price[good], lead[good]

def read_tsf(path):
    """Minimal reader for the Monash .tsf format: lines after @data look like  name:start:v1,v2,..."""
    rows, started = [], False
    for line in open(path, encoding='utf-8', errors='ignore'):
        line = line.strip()
        if not line or line.startswith('#'): continue
        if line.lower().startswith('@data'): started = True; continue
        if not started: continue
        vals = line.split(':')[-1].split(',')
        rows.append([0.0 if v in ('?', '') else float(v) for v in vals])
    n = min(len(r) for r in rows)
    return np.array([r[:n] for r in rows])

def load_carparts():
    urllib.request.urlretrieve(CAR_URL, 'carparts.zip')
    with zipfile.ZipFile('carparts.zip') as z:
        name = [f for f in z.namelist() if f.endswith('.tsf')][0]
        z.extract(name, 'carparts_data')
    Y = read_tsf(os.path.join('carparts_data', name))
    return Y, np.ones(len(Y)), np.full(len(Y), float(LEAD_TIME))

def make_synthetic(n_per=100, n=51, seed=SEED):
    """FALLBACK ONLY. 4 clearly separated demand types."""
    r = np.random.default_rng(seed); Y, lab = [], []
    for _ in range(n_per): Y.append(r.poisson(20, n)); lab.append('Smooth')
    for _ in range(n_per): Y.append(np.round(r.lognormal(2.5, 1.0, n))); lab.append('Erratic')
    for _ in range(n_per): Y.append((r.random(n) < 0.3) * (1 + r.poisson(6, n))); lab.append('Intermittent')
    for _ in range(n_per): Y.append((r.random(n) < 0.2) * np.round(r.lognormal(2.2, 1.0, n))); lab.append('Lumpy')
    Y = np.array(Y, dtype=float)
    return Y, np.ones(len(Y)), np.full(len(Y), float(LEAD_TIME)), np.array(lab)

USING_REAL, true_labels = True, None
order = [DATASET] + [d for d in ['RAF', 'OIL', 'CARPARTS'] if d != DATASET]
Y_all = None
for name_ in order:
    try:
        Y_all, price_all, lead_all = {'RAF': load_raf, 'OIL': load_oil, 'CARPARTS': load_carparts}[name_]()
        DATA_NAME = name_; print(f'Loaded REAL {name_} data:', Y_all.shape); break
    except Exception as e:
        print(f'{name_} download failed:', str(e)[:120])
if Y_all is None:
    Y_all, price_all, lead_all, true_labels = make_synthetic(); USING_REAL = False; DATA_NAME = 'SYNTHETIC'
    print('!!! USING SYNTHETIC FALLBACK DATA - do NOT report these results as real:', Y_all.shape)

neg = int((Y_all < 0).sum())
if neg:   # e.g. returns / adjustments in OIL: demand cannot be negative, so treat as zero demand (document this in the report)
    print(f'Note: {neg} negative demand cells ({100 * neg / Y_all.size:.3f}% of all cells) set to 0')
    Y_all = np.maximum(Y_all, 0)

# Remove system-wide anomaly months (e.g. a mass booking/adjustment): a month where far more items than usual
# show demand is a recording event, not real demand. Flagged automatically and reported.
share = (Y_all > 0).mean(0); bad = np.where(share > 3 * np.median(share))[0]
if len(bad):
    print(f'Anomaly check: removed month index(es) {bad.tolist()} - share of items with demand {np.round(100 * share[bad], 1).tolist()}% vs median {100 * np.median(share):.1f}%')
    Y_all = np.delete(Y_all, bad, axis=1)
else:
    print('Anomaly check: no system-wide anomaly months found')

n = Y_all.shape[1]; n_train = n - TEST_MONTHS
print(f'{DATA_NAME}: {n} months total -> train: {n_train}, test: {TEST_MONTHS}')

keep = np.where((Y_all[:, :n_train] > 0).sum(1) >= MIN_DEMANDS)[0]
if SAMPLE_N and len(keep) > SAMPLE_N:
    keep = rng.choice(keep, SAMPLE_N, replace=False)
Y = Y_all[keep]; N = len(Y); price = price_all[keep]; lead_base = lead_all[keep]
if true_labels is not None: true_labels = true_labels[keep]
print('Series used:', N, '| zero share in data: %.0f%%' % (100 * (Y == 0).mean()))
print('Lead time (months) median/max: %.0f / %.0f | median price: %.2f' % (np.median(lead_base), lead_base.max(), np.median(price)))

## 2. Classify demand (ADI / CV²) — using TRAIN data only
Standard Syntetos–Boylan cut-offs: ADI = 1.32, CV² = 0.49. (ADI = periods ÷ number of demand months; CV² = squared coefficient of variation of non-zero demand sizes.)

In [ ]:
def classify(y):
    nz = y[y > 0]
    adi = len(y) / len(nz)
    cv2 = (nz.std() / nz.mean()) ** 2
    if   adi < 1.32 and cv2 < 0.49: c = 'Smooth'
    elif adi < 1.32:                c = 'Erratic'
    elif cv2 < 0.49:                c = 'Intermittent'
    else:                           c = 'Lumpy'
    return c, adi, cv2

cls = pd.DataFrame([classify(Y[i, :n_train]) for i in range(N)], columns=['category', 'ADI', 'CV2'])
category = cls['category'].values
print(cls['category'].value_counts())

if true_labels is not None:   # sanity check only possible with synthetic data
    print('\nClassifier check (synthetic only):')
    print(pd.crosstab(pd.Series(true_labels, name='true'), pd.Series(category, name='predicted')))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
cls['category'].value_counts().plot.bar(ax=ax[0], color='steelblue', title='Demand categories')
ax[1].scatter(cls['ADI'], cls['CV2'], s=6, alpha=.4); ax[1].axvline(1.32, c='r', ls='--'); ax[1].axhline(0.49, c='r', ls='--')
ax[1].set_xscale('log'); ax[1].set_xlabel('ADI (log)'); ax[1].set_ylabel('CV²'); ax[1].set_title('ADI–CV² plane')
plt.tight_layout(); plt.savefig(f'fig1_classification_{DATA_NAME}.png', dpi=150); plt.show()

## 3. Forecasting methods
Every function returns `f[t]` = forecast for month **t+1** using data up to month **t** only (no peeking). One pass therefore gives forecasts for *every* test month (rolling-origin evaluation).

- **MA** – 6-month moving average (baseline)
- **Croston** – smooths demand size and interval separately
- **SBA** – Croston with bias correction (1 − α/2)
- **TSB** – smooths demand *probability* every month (handles obsolescence)
- **LightGBM** – one global model across all parts (Tweedie loss, suitable for zero-heavy data)

In [ ]:
def f_ma(y, w=6):
    f = np.zeros(len(y))
    for t in range(len(y)): f[t] = y[max(0, t - w + 1): t + 1].mean()
    return f

def f_croston(y, alpha=0.1, sba=False):
    f = np.zeros(len(y)); z = p = None; q = 1
    for t, d in enumerate(y):
        if d > 0:
            if z is None: z, p = d, q
            else: z += alpha * (d - z); p += alpha * (q - p)
            q = 1
        else: q += 1
        if z is not None: f[t] = (1 - alpha / 2 if sba else 1.0) * z / p
    return f

def f_tsb(y, alpha=0.1, beta=0.1):
    f = np.zeros(len(y)); z = p = None
    for t, d in enumerate(y):
        if z is None:
            if d > 0: z, p = d, 1.0 / (t + 1)
        else:
            p += beta * ((d > 0) - p)
            if d > 0: z += alpha * (d - z)
        if z is not None: f[t] = p * z
    return f

# ---- global LightGBM -------------------------------------------------------
LAGS = 6
def make_features(Ys):
    """Features at month t (known at t) for predicting t+1. Ys = demand scaled per series."""
    N_, n_ = Ys.shape; F = np.full((N_, n_, LAGS + 5), np.nan)
    since = np.zeros((N_, n_)); c = np.zeros(N_)
    for t in range(n_):
        c = np.where(Ys[:, t] > 0, 0, c + 1); since[:, t] = np.minimum(c, 24)
    for t in range(LAGS - 1, n_):
        for k in range(LAGS): F[:, t, k] = Ys[:, t - k]
        F[:, t, LAGS]     = Ys[:, t - 2: t + 1].mean(1)
        F[:, t, LAGS + 1] = Ys[:, t - 5: t + 1].mean(1)
        F[:, t, LAGS + 2] = (Ys[:, t - 5: t + 1] > 0).mean(1)
        F[:, t, LAGS + 3] = since[:, t]
        F[:, t, LAGS + 4] = t % 12
    return F

def f_lgbm(Yfull, fit_end):
    """Train ONLY on targets before fit_end; return forecasts f[:, t] for all t."""
    scale = np.maximum(Yfull[:, :fit_end].mean(1), 0.05)
    Ys = Yfull / scale[:, None]
    F = make_features(Ys)
    Xs, ys = [], []
    for t in range(LAGS - 1, fit_end - 1):          # target index t+1 <= fit_end-1
        Xs.append(F[:, t, :]); ys.append(Ys[:, t + 1])
    X, y = np.vstack(Xs), np.concatenate(ys)
    m = lgb.LGBMRegressor(objective='tweedie', tweedie_variance_power=1.3, n_estimators=200,
                          learning_rate=0.05, num_leaves=15, min_child_samples=20, verbose=-1, random_state=SEED)
    m.fit(X, y)
    out = np.zeros(Yfull.shape)
    for t in range(LAGS - 1, Yfull.shape[1]):
        out[:, t] = np.maximum(m.predict(F[:, t, :]), 0) * scale
    return out

print('Forecasting functions ready')

## 4. Run forecasts
Two versions of every forecast are kept (this is what makes the comparison fair):
- **`FC` (calibration version):** LightGBM trained only on months before the last `CAL_MONTHS` of train. Used to measure forecast-error spread (σ) and to choose the Adaptive method, with no leakage.
- **`F` (final version):** trained on all train months. Used for the test period.

The other methods are the same in both (they only use past data).

In [ ]:
F, FC = {}, {}
F['MA']      = np.array([f_ma(Y[i]) for i in range(N)]);                  FC['MA'] = F['MA']
F['Croston'] = np.array([f_croston(Y[i]) for i in range(N)]);             FC['Croston'] = F['Croston']
F['SBA']     = np.array([f_croston(Y[i], sba=True) for i in range(N)]);   FC['SBA'] = F['SBA']
F['TSB']     = np.array([f_tsb(Y[i]) for i in range(N)]);                 FC['TSB'] = F['TSB']
# f_lgbm trains ONLY on targets before fit_end and scales using months before fit_end, so passing the full
# series is safe: test months are used only as observed lags when forecasting the next month (rolling origin).
F['LightGBM']  = f_lgbm(Y, n_train)
FC['LightGBM'] = f_lgbm(Y, n_train - CAL_MONTHS)

base_methods = ['MA', 'Croston', 'SBA', 'TSB', 'LightGBM']
cal = slice(n_train - CAL_MONTHS, n_train)          # calibration target months
tst = slice(n_train, n)                             # test target months

def err(Fm, sl):   # forecast for month j is f[j-1]
    return Y[:, sl] - Fm[:, sl.start - 1: sl.stop - 1]

# scale factors (from TRAIN only) for scaled metrics
naive_mae  = np.abs(np.diff(Y[:, :n_train], axis=1)).mean(1)
naive_rmse = np.sqrt((np.diff(Y[:, :n_train], axis=1) ** 2).mean(1))

# --- Adaptive: for each demand category, pick the method with the lowest CALIBRATION error ---
cal_rmsse = {m: np.sqrt((err(FC[m], cal) ** 2).mean(1)) / naive_rmse for m in base_methods}
choice = {}
for c in ['Smooth', 'Erratic', 'Intermittent', 'Lumpy']:
    idx = category == c
    if idx.sum() == 0: continue
    choice[c] = min(base_methods, key=lambda m: cal_rmsse[m][idx].mean())
print('Adaptive routing (chosen on calibration window, no test data used):', choice)
for store in (F, FC):
    store['Adaptive'] = np.array([ (F if store is F else FC)[choice[category[i]]][i] for i in range(N)])
METHODS = base_methods + ['Adaptive']

## 5. Accuracy evaluation (rolling, all 12 test months, every part)
- **MASE** – error relative to a naive forecast (scale-free, works with zeros; unlike MAPE)
- **RMSSE** – like MASE but squared errors (rewards getting the *mean* right; matters for stock decisions)
- **Scaled bias** – average over-/under-forecast relative to average demand (positive = over-forecast)

Note: MAE-type metrics reward forecasting 0 on intermittent data, so read MASE together with RMSSE and bias.

In [ ]:
rows = []
for m in METHODS:
    e = err(F[m], tst)
    mase  = np.abs(e).mean(1) / naive_mae
    rmsse = np.sqrt((e ** 2).mean(1)) / naive_rmse
    bias  = (-e).mean(1) / np.maximum(Y[:, :n_train].mean(1), 1e-9)
    rows.append({'method': m, 'MASE': mase.mean(), 'RMSSE': rmsse.mean(), 'bias': bias.mean()})
acc = pd.DataFrame(rows).set_index('method').round(3)
print(acc.sort_values('RMSSE'))

# by demand category
bycat = {}
for m in METHODS:
    e = err(F[m], tst); r = np.sqrt((e ** 2).mean(1)) / naive_rmse
    bycat[m] = pd.Series(r).groupby(category).mean()
print('\nRMSSE by demand category (lower = better):')
print(pd.DataFrame(bycat).round(3))

# Is the difference real? Paired test per series vs SBA
sba_m = np.abs(err(F['SBA'], tst)).mean(1) / naive_mae
print('\nWilcoxon paired test on per-series MASE vs SBA (p<0.05 = difference unlikely to be chance):')
for m in METHODS:
    if m == 'SBA': continue
    mm = np.abs(err(F[m], tst)).mean(1) / naive_mae
    p = wilcoxon(mm, sba_m).pvalue if np.any(mm != sba_m) else 1.0
    print(f'  {m:9s} vs SBA: mean diff {np.mean(mm - sba_m):+.3f}, p = {p:.4f}')

acc.sort_values('RMSSE')[['MASE', 'RMSSE']].plot.barh(figsize=(8, 4), title='Forecast accuracy (lower = better)')
plt.tight_layout(); plt.savefig(f'fig2_accuracy_{DATA_NAME}.png', dpi=150); plt.show()

## 6. Inventory simulation (month by month, per item)
For each part and method (using the item's **own lead time L** and **price**):
1. Start in steady state: safety stock on hand, and one expected month of demand already on order for each of the next L months.
2. Each month: receive arrivals → serve demand (unmet demand is **lost** and penalised) → pay holding on leftover stock → order up to  
   **S = forecast × L + z × σ × √L**, where σ = forecast-error spread measured on the calibration window (train data only).
3. Costs: holding = price × `HOLD_RATE`/12 per unit-month; shortage = `SHORT_MULT` × price per unit short.
4. Two stocking policies:
   - **Flat:** same z for every part (`Z_FLAT`).
   - **Criticality-aware:** z from the cost ratio, `z = Φ⁻¹(s / (s + h))` (newsvendor rule) – critical parts get more safety stock.

⚠ Criticality classes and the cost multipliers are **assumptions** (not in the data). Section 8 tests how much the conclusions depend on them.
Note: a few expensive items dominate total £ cost, so section 6 also tests cost differences item by item.

In [ ]:
classes = np.array(rng.choice(list(CLASS_SHARE), size=N, p=list(CLASS_SHARE.values())))
h_i  = price * HOLD_RATE / 12.0                      # holding cost per unit-month
mult = np.array([SHORT_MULT[c] for c in classes])

def make_costs(short_scale=1.0, lead_scale=1.0):
    s_unit = mult * short_scale * price
    Ls = np.maximum(1, np.round(lead_base * lead_scale)).astype(int)
    z_crit = norm.ppf(s_unit / (s_unit + h_i))
    return s_unit, Ls, z_crit

SIGMA = {m: np.sqrt((err(FC[m], cal) ** 2).mean(1)) for m in METHODS}   # forecast-error spread per item (train data only)

def simulate(i, m, z, s_unit, Ls):
    L = int(Ls[i]); sigma = SIGMA[m][i]
    f = F[m][i]; y = Y[i, tst]; h = h_i[i]; ss = z * sigma * np.sqrt(L)
    on_hand = ss; pipe = [f[n_train - 1]] * L          # steady-state start (no test info used)
    hold = short = unmet = dem = 0.0; stockout_months = 0
    for j in range(len(y)):
        on_hand += pipe.pop(0)
        sold = min(on_hand, y[j]); miss = y[j] - sold; on_hand -= sold
        hold += on_hand * h; short += miss * s_unit[i]; unmet += miss; dem += y[j]; stockout_months += miss > 0
        S = f[n_train + j] * L + z * sigma * np.sqrt(L)
        pipe.append(max(0.0, S - (on_hand + sum(pipe))))
    return hold, short, unmet, dem, stockout_months

def run_policy(policy, short_scale=1.0, lead_scale=1.0, methods=None):
    s_unit, Ls, z_crit = make_costs(short_scale, lead_scale)
    out, items = [], {}
    for m in (methods or METHODS):
        r = np.array([simulate(i, m, Z_FLAT if policy == 'Flat' else z_crit[i], s_unit, Ls) for i in range(N)])
        items[m] = r[:, 0] + r[:, 1]
        hold, short, unmet, dem, so = r.sum(0)
        out.append({'policy': policy, 'method': m, 'holding': hold, 'shortage': short, 'total_cost': hold + short,
                    'fill_rate_%': 100 * (1 - unmet / dem), 'stockout_months': int(so)})
    return pd.DataFrame(out), items

r_flat, items_flat = run_policy('Flat')
r_crit, items_crit = run_policy('Criticality-aware')
res = pd.concat([r_flat, r_crit]).round(1)
print(res.sort_values(['policy', 'total_cost']).to_string(index=False))

print('\nPaired Wilcoxon on per-item total cost vs SBA (flat policy):')
for m in METHODS:
    if m == 'SBA': continue
    d = items_flat[m] - items_flat['SBA']
    p = wilcoxon(items_flat[m], items_flat['SBA']).pvalue if np.any(d != 0) else 1.0
    print(f'  {m:9s} vs SBA: median item diff {np.median(d):+.3f}, mean {d.mean():+.2f}, p = {p:.4f}')

## 7. The research answer: accuracy vs cost

In [ ]:
flat = res[res.policy == 'Flat'].set_index('method')
comp = acc[['MASE', 'RMSSE', 'bias']].join(flat[['total_cost', 'fill_rate_%']])
comp['accuracy_rank (RMSSE)'] = comp['RMSSE'].rank().astype(int)
comp['cost_rank'] = comp['total_cost'].rank().astype(int)
print(comp.sort_values('cost_rank'))
rho = comp['RMSSE'].rank().corr(comp['total_cost'].rank())
print(f'\nRank correlation between accuracy and cost: {rho:.2f}  (1 = most accurate is always cheapest)')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m in comp.index:
    ax[0].scatter(comp.loc[m, 'RMSSE'], comp.loc[m, 'total_cost'], s=80); ax[0].annotate(m, (comp.loc[m, 'RMSSE'], comp.loc[m, 'total_cost']))
ax[0].set_xlabel('RMSSE (accuracy, lower = better)'); ax[0].set_ylabel('Total cost (flat policy)'); ax[0].set_title('Accuracy vs inventory cost')
piv = res.pivot(index='method', columns='policy', values='total_cost'); piv.plot.bar(ax=ax[1], title='Total cost: flat vs criticality-aware'); ax[1].set_ylabel('Total cost')
plt.tight_layout(); plt.savefig(f'fig3_accuracy_vs_cost_{DATA_NAME}.png', dpi=150); plt.show()
res.to_csv(f'results_inventory_{DATA_NAME}.csv', index=False); acc.to_csv(f'results_accuracy_{DATA_NAME}.csv')

## 7b. Fair comparison: cost at the SAME service level
Section 6 compares methods at one z, but methods then end up at different fill rates (a method that over-forecasts "buys" a higher fill rate). Here every method is run at many z values to draw its **cost-vs-fill-rate curve**, and costs are compared at the **same fill rate**. This separates *forecast quality* from *how much stock a method happens to carry*.

In [ ]:
Z_GRID = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
s_unit1, Ls1, _ = make_costs()
curve = {m: [] for m in base_methods}
for m in base_methods:
    for zz in Z_GRID:
        r = np.array([simulate(i, m, zz, s_unit1, Ls1) for i in range(N)]).sum(0)
        curve[m].append((100 * (1 - r[2] / r[3]), r[0] + r[1], zz))
fig, ax = plt.subplots(figsize=(8, 5))
for m in base_methods:
    a = np.array(curve[m]); ax.plot(a[:, 0], a[:, 1], marker='o', label=m)
ax.set_xlabel('Fill rate (%)'); ax.set_ylabel('Total cost'); ax.set_title('Cost vs service level (lower-right is better)'); ax.legend()
plt.tight_layout(); plt.savefig(f'fig4_cost_vs_fillrate_{DATA_NAME}.png', dpi=150); plt.show()

lo = max(min(a for a, _, _ in curve[m]) for m in base_methods); hi = min(max(a for a, _, _ in curve[m]) for m in base_methods)
targets = np.round(np.linspace(lo + 0.15 * (hi - lo), hi - 0.05 * (hi - lo), 3), 1)
rows = {}
for m in base_methods:
    a = np.array(sorted(curve[m]))
    rows[m] = [np.interp(t, a[:, 0], a[:, 1]) for t in targets]
eq = pd.DataFrame(rows, index=[f'cost @ fill {t}%' for t in targets]).T.round(0)
eq['mean_rank'] = eq.rank().mean(1).round(2)
print('Total cost at the SAME fill rate (lower = better):'); print(eq.sort_values('mean_rank'))
print('\nCompare with accuracy ranking (RMSSE):', list(acc.loc[base_methods, 'RMSSE'].sort_values().index))
eq.to_csv(f'results_equal_service_{DATA_NAME}.csv')

## 8. Sensitivity check (do conclusions survive different assumptions?)
Re-runs flat vs criticality-aware for the **best-cost method** while scaling the shortage costs (×0.5, ×1, ×2) and the lead times (×0.5, ×1, ×1.5). If the conclusion flips, say so in the report.

In [ ]:
best = flat['total_cost'].idxmin()
print('Best method under flat policy:', best)
rows = []
for sc in [0.5, 1, 2]:
    for ls in [0.5, 1, 1.5]:
        row = {'shortage_cost_x': sc, 'lead_time_x': ls}
        for policy in ['Flat', 'Criticality-aware']:
            r, _ = run_policy(policy, sc, ls, methods=[best]); row[policy] = r['total_cost'].iloc[0]
        rows.append(row)
sens = pd.DataFrame(rows).set_index(['shortage_cost_x', 'lead_time_x']).round(0)
sens['saving_%'] = (100 * (1 - sens['Criticality-aware'] / sens['Flat'])).round(1)
print(sens)
import json
summary = {'dataset': DATA_NAME, 'real_data': USING_REAL, 'n_series': int(N), 'n_months': int(n), 'train': int(n_train), 'test': int(TEST_MONTHS),
           'categories': pd.Series(category).value_counts().to_dict(), 'accuracy': acc.to_dict('index'),
           'inventory': res.to_dict('records'), 'rank_corr_acc_cost': float(rho), 'equal_service': eq.to_dict('index'),
           'sensitivity_best_method': best, 'sensitivity': sens.reset_index().to_dict('records')}
json.dump(summary, open(f'summary_{DATA_NAME}.json', 'w'), indent=1, default=float)
print('Saved summary_%s.json' % DATA_NAME)

## 9. What to write in your report (fill with YOUR numbers)
1. **Data & limitations:** RAF (main) and car parts (second check), N series used, train/test split, assumed criticality, shortage multipliers and holding rate. Cite the RAF source repository and the thesis it comes from.
2. **Accuracy:** table from section 5 + Wilcoxon results. State which differences are statistically meaningful.
3. **Cost:** table from section 6. Which method is cheapest? Is it the most accurate one (section 7 correlation)?
4. **Criticality:** saving % from section 8, and under which assumptions it holds.
5. **Conclusion:** answer the research question honestly. If simple methods (SBA/TSB/MA) match LightGBM, say so — that is a valid finding on intermittent data.
6. **Future work:** real failure / condition data (RUL) as a demand driver; prediction intervals (conformal); real price and downtime-cost data.